# V4 Intent Classifier vs LLM Slot Extraction Comparison

This notebook compares two NLU approaches for Indonesian e-commerce chatbot:

1. **V4 Intent Classifier**: DistilBERT fine-tuned on 2,400 samples (6 intents)
2. **LLM Slot Extraction**: Qwen3-8B with few-shot prompting (zero training)

## Prerequisites

1. V4 model trained and saved to `models/intent_classifier_v4/`
2. llama-server running: `./scripts/start_llm_server.sh`

In [ ]:
# Cell 1: Setup and Imports
import os
import sys
import json
import time
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from typing import Dict, Any, List, Tuple

# Project paths
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python version: {sys.version}")

## Label Mapping

| V4 Intent | LLM Task |
|-----------|----------|
| order_status | check_order |
| payment_info | ask_payment |
| product_price | ask_price |
| product_stock | check_stock |
| product_description | product_info |
| out_of_scope | out_of_scope |

In [ ]:
# Cell 2: Label Mapping

V4_TO_LLM = {
    "order_status": "check_order",
    "payment_info": "ask_payment",
    "product_price": "ask_price",
    "product_stock": "check_stock",
    "product_description": "product_info",
    "out_of_scope": "out_of_scope"
}

LLM_TO_V4 = {v: k for k, v in V4_TO_LLM.items()}

print("Label Mapping:")
for v4, llm in V4_TO_LLM.items():
    print(f"  {v4:<20} <-> {llm}")

In [ ]:
# Cell 3: Load V4 Intent Classifier
from transformers import pipeline

V4_MODEL_PATH = PROJECT_ROOT / "models" / "intent_classifier_v4"

print(f"Loading V4 model from: {V4_MODEL_PATH}")

# Check if model exists
if not V4_MODEL_PATH.exists():
    raise FileNotFoundError(f"V4 model not found at {V4_MODEL_PATH}")

# Load classifier
v4_classifier = pipeline(
    "text-classification",
    model=str(V4_MODEL_PATH),
    device=-1  # CPU
)

# Load label mapping
with open(V4_MODEL_PATH / "label_mapping.json", 'r') as f:
    v4_label_mapping = json.load(f)

print(f"V4 Model loaded!")
print(f"  Labels: {list(v4_label_mapping['label2id'].keys())}")
print(f"  Version: {v4_label_mapping.get('version', 'unknown')}")

In [ ]:
# Cell 4: Check LLM Server
from src.llm_slot_extractor import SlotExtractor

LLM_SERVER_URL = "http://localhost:8080"

llm_extractor = SlotExtractor(server_url=LLM_SERVER_URL)

if llm_extractor.is_server_running():
    print(f"LLM Server is running at {LLM_SERVER_URL}")
else:
    print("ERROR: LLM Server is NOT running!")
    print("")
    print("Start the server in a separate terminal:")
    print("  ./scripts/start_llm_server.sh")
    raise ConnectionError("LLM server not running")

In [ ]:
# Cell 5: Load Evaluation Dataset

EVAL_DATASET_PATH = PROJECT_ROOT / "data" / "evaluation" / "llm_slot_extraction_eval.json"

with open(EVAL_DATASET_PATH, 'r', encoding='utf-8') as f:
    eval_dataset = json.load(f)

queries = eval_dataset["queries"]
print(f"Loaded {len(queries)} evaluation queries")

# Show distribution
categories = [q.get('category', 'unknown') for q in queries]
from collections import Counter
print(f"\nCategory distribution:")
for cat, count in Counter(categories).most_common():
    print(f"  {cat}: {count}")

In [ ]:
# Cell 6: Define Comparison Function

def compare_single_query(query: str, expected: Dict) -> Dict:
    """
    Run both V4 and LLM on a single query and compare results.
    """
    expected_task = expected.get("task", "out_of_scope")
    expected_intent = LLM_TO_V4.get(expected_task, "out_of_scope")
    
    # Run V4
    v4_start = time.time()
    v4_result = v4_classifier(query)[0]
    v4_latency = time.time() - v4_start
    
    v4_intent = v4_result['label']
    v4_confidence = v4_result['score']
    v4_task = V4_TO_LLM.get(v4_intent, "out_of_scope")
    v4_correct = (v4_task.lower() == expected_task.lower())
    
    # Run LLM
    llm_start = time.time()
    llm_result = llm_extractor.extract(query)
    llm_latency = time.time() - llm_start
    
    llm_task = llm_result.task
    llm_confidence = llm_result.confidence
    llm_error = llm_result.error
    llm_correct = (llm_task.lower() == expected_task.lower()) if not llm_error else False
    
    return {
        "query": query,
        "expected_task": expected_task,
        "expected_intent": expected_intent,
        
        # V4 results
        "v4_intent": v4_intent,
        "v4_task": v4_task,
        "v4_confidence": v4_confidence,
        "v4_latency_ms": v4_latency * 1000,
        "v4_correct": v4_correct,
        
        # LLM results
        "llm_task": llm_task,
        "llm_confidence": llm_confidence,
        "llm_latency_s": llm_latency,
        "llm_entities": llm_result.entities,
        "llm_multi_intent": llm_result.multi_intent,
        "llm_correct": llm_correct,
        "llm_error": llm_error,
        
        # Comparison
        "both_correct": v4_correct and llm_correct,
        "both_wrong": not v4_correct and not llm_correct,
        "v4_only_correct": v4_correct and not llm_correct,
        "llm_only_correct": not v4_correct and llm_correct
    }

print("Comparison function defined!")

In [ ]:
# Cell 7: Run Full Comparison

print(f"Running comparison on {len(queries)} queries...")
print("=" * 70)

results = []

for i, item in enumerate(queries):
    query = item["query"]
    expected = item["expected"]
    
    result = compare_single_query(query, expected)
    results.append(result)
    
    # Progress indicator
    v4_status = "✓" if result["v4_correct"] else "✗"
    llm_status = "✓" if result["llm_correct"] else ("ERR" if result["llm_error"] else "✗")
    
    print(f"[{i+1:2}/{len(queries)}] {query[:50]:<50}")
    print(f"         V4: {result['v4_intent']:<20} [{v4_status}] | LLM: {result['llm_task']:<15} [{llm_status}]")

print("\n" + "=" * 70)
print(f"Comparison complete!")

In [ ]:
# Cell 8: Calculate Metrics

total = len(results)

# V4 metrics
v4_correct = sum(1 for r in results if r["v4_correct"])
v4_latencies = [r["v4_latency_ms"] for r in results]

# LLM metrics
llm_errors = sum(1 for r in results if r["llm_error"])
llm_valid = total - llm_errors
llm_correct = sum(1 for r in results if r["llm_correct"])
llm_latencies = [r["llm_latency_s"] for r in results if not r["llm_error"]]

# Comparison metrics
both_correct = sum(1 for r in results if r["both_correct"])
both_wrong = sum(1 for r in results if r["both_wrong"])
v4_only = sum(1 for r in results if r["v4_only_correct"])
llm_only = sum(1 for r in results if r["llm_only_correct"])

metrics = {
    "total_queries": total,
    
    "v4": {
        "accuracy": v4_correct / total,
        "correct": v4_correct,
        "latency_mean_ms": np.mean(v4_latencies),
        "latency_min_ms": np.min(v4_latencies),
        "latency_max_ms": np.max(v4_latencies),
    },
    
    "llm": {
        "accuracy_overall": llm_correct / total,
        "accuracy_valid": llm_correct / llm_valid if llm_valid > 0 else 0,
        "correct": llm_correct,
        "valid_responses": llm_valid,
        "errors": llm_errors,
        "error_rate": llm_errors / total,
        "latency_mean_s": np.mean(llm_latencies) if llm_latencies else 0,
        "latency_min_s": np.min(llm_latencies) if llm_latencies else 0,
        "latency_max_s": np.max(llm_latencies) if llm_latencies else 0,
    },
    
    "comparison": {
        "both_correct": both_correct,
        "both_correct_pct": both_correct / total,
        "both_wrong": both_wrong,
        "v4_only_correct": v4_only,
        "llm_only_correct": llm_only,
        "agreement_rate": (both_correct + both_wrong) / total,
    }
}

print("Metrics calculated!")

In [ ]:
# Cell 9: Print Summary

print("\n" + "=" * 70)
print("COMPARISON SUMMARY: V4 Intent Classifier vs LLM Slot Extraction")
print("=" * 70)

print(f"\nTotal Queries: {metrics['total_queries']}")

print("\n" + "-" * 40)
print("V4 INTENT CLASSIFIER (DistilBERT)")
print("-" * 40)
v4 = metrics["v4"]
print(f"  Accuracy: {v4['accuracy']:.1%} ({v4['correct']}/{total})")
print(f"  Latency:  {v4['latency_mean_ms']:.1f}ms avg ({v4['latency_min_ms']:.1f}-{v4['latency_max_ms']:.1f}ms)")

print("\n" + "-" * 40)
print("LLM SLOT EXTRACTION (Qwen3-8B)")
print("-" * 40)
llm = metrics["llm"]
print(f"  Accuracy (all):   {llm['accuracy_overall']:.1%} ({llm['correct']}/{total})")
print(f"  Accuracy (valid): {llm['accuracy_valid']:.1%} ({llm['correct']}/{llm['valid_responses']})")
print(f"  Error Rate:       {llm['error_rate']:.1%} ({llm['errors']} errors)")
print(f"  Latency:          {llm['latency_mean_s']:.2f}s avg ({llm['latency_min_s']:.2f}-{llm['latency_max_s']:.2f}s)")

print("\n" + "-" * 40)
print("HEAD-TO-HEAD COMPARISON")
print("-" * 40)
comp = metrics["comparison"]
print(f"  Both Correct:      {comp['both_correct_pct']:.1%} ({comp['both_correct']})")
print(f"  Both Wrong:        {comp['both_wrong']}/{total}")
print(f"  V4 Only Correct:   {comp['v4_only_correct']}/{total}")
print(f"  LLM Only Correct:  {comp['llm_only_correct']}/{total}")
print(f"  Agreement Rate:    {comp['agreement_rate']:.1%}")

In [ ]:
# Cell 10: By Category Analysis

print("\n" + "-" * 40)
print("BY CATEGORY")
print("-" * 40)

# Group by category
category_results = {}
for i, r in enumerate(results):
    cat = queries[i].get('category', 'unknown')
    if cat not in category_results:
        category_results[cat] = {"total": 0, "v4_correct": 0, "llm_correct": 0, "llm_valid": 0}
    category_results[cat]["total"] += 1
    if r["v4_correct"]:
        category_results[cat]["v4_correct"] += 1
    if not r["llm_error"]:
        category_results[cat]["llm_valid"] += 1
        if r["llm_correct"]:
            category_results[cat]["llm_correct"] += 1

print(f"{'Category':<18} {'V4':>10} {'LLM (valid)':>12} {'Total':>8}")
print("-" * 50)
for cat, m in sorted(category_results.items()):
    v4_acc = f"{m['v4_correct']/m['total']:.0%}"
    llm_acc = f"{m['llm_correct']/m['llm_valid']:.0%}" if m['llm_valid'] > 0 else "N/A"
    print(f"{cat:<18} {v4_acc:>10} {llm_acc:>12} {m['total']:>8}")

In [ ]:
# Cell 11: Feature Comparison Table

print("\n" + "-" * 40)
print("FEATURE COMPARISON")
print("-" * 40)

v4_lat = metrics['v4']['latency_mean_ms']
llm_lat = metrics['llm']['latency_mean_s']

feature_table = f"""
| Feature              | V4 Classifier    | LLM Slot Extraction |
|----------------------|------------------|---------------------|
| Model                | DistilBERT (66M) | Qwen3-8B (8B)       |
| Model Size           | ~250MB           | ~5GB (Q4_K_M)       |
| Training Data        | 2,400 samples    | Zero (prompt only)  |
| Task Accuracy        | {metrics['v4']['accuracy']:.1%}            | {metrics['llm']['accuracy_valid']:.1%} (on valid)    |
| Response Rate        | 100%             | {(1-metrics['llm']['error_rate']):.1%}              |
| Latency              | {v4_lat:.0f}ms             | {llm_lat:.2f}s              |
| Multi-intent         | No               | Yes (100%)          |
| Entity Extraction    | No               | Yes (88%)           |
| Clarification        | No               | Yes (98%)           |
| Flexibility          | Fixed 6 intents  | Unlimited via prompt|
"""

print(feature_table)

In [ ]:
# Cell 12: Error Analysis - Where Each Model Fails

print("\n" + "=" * 70)
print("ERROR ANALYSIS")
print("=" * 70)

# V4 only correct (LLM failed)
print("\n--- V4 CORRECT, LLM WRONG ---")
v4_wins = [r for r in results if r["v4_only_correct"]]
for r in v4_wins[:5]:  # Show first 5
    print(f"  Query: {r['query']}")
    print(f"  Expected: {r['expected_task']} | V4: {r['v4_task']} ✓ | LLM: {r['llm_task']} ✗")
    if r['llm_error']:
        print(f"  LLM Error: {r['llm_error']}")
    print()

# LLM only correct (V4 failed)
print("\n--- LLM CORRECT, V4 WRONG ---")
llm_wins = [r for r in results if r["llm_only_correct"]]
for r in llm_wins[:5]:  # Show first 5
    print(f"  Query: {r['query']}")
    print(f"  Expected: {r['expected_task']} | V4: {r['v4_task']} ✗ | LLM: {r['llm_task']} ✓")
    print()

# Both wrong
print("\n--- BOTH WRONG ---")
both_fail = [r for r in results if r["both_wrong"]]
for r in both_fail[:5]:  # Show first 5
    print(f"  Query: {r['query']}")
    print(f"  Expected: {r['expected_task']} | V4: {r['v4_task']} | LLM: {r['llm_task']}")
    print()

In [ ]:
# Cell 13: LLM Unique Features Demo

print("\n" + "=" * 70)
print("LLM UNIQUE FEATURES (Not Available in V4)")
print("=" * 70)

# Entity extraction examples
print("\n--- ENTITY EXTRACTION ---")
entity_examples = [r for r in results if r['llm_entities'] and any(v for v in r['llm_entities'].values() if v is not None)]
for r in entity_examples[:5]:
    print(f"  Query: {r['query']}")
    print(f"  Entities: {r['llm_entities']}")
    print()

# Multi-intent examples
print("\n--- MULTI-INTENT DETECTION ---")
multi_intent_examples = [r for r in results if r['llm_multi_intent']]
for r in multi_intent_examples[:5]:
    print(f"  Query: {r['query']}")
    print(f"  Primary: {r['llm_task']} | Secondary: {r['llm_multi_intent']}")
    print()

In [ ]:
# Cell 14: Save Results

OUTPUT_DIR = PROJECT_ROOT / "evaluation"
OUTPUT_DIR.mkdir(exist_ok=True)

# Save full results
output = {
    "timestamp": datetime.now().isoformat(),
    "metrics": metrics,
    "by_category": category_results,
    "results": results
}

output_path = OUTPUT_DIR / "v4_vs_llm_comparison.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False, default=str)

print(f"Results saved to: {output_path}")

# Save thesis table as markdown
md_path = OUTPUT_DIR / "v4_vs_llm_comparison.md"
with open(md_path, 'w', encoding='utf-8') as f:
    f.write(f"# V4 vs LLM Comparison Results\n\n")
    f.write(f"Generated: {datetime.now().isoformat()}\n\n")
    f.write(feature_table)
    f.write("\n\n## Summary\n\n")
    f.write(f"- V4 Accuracy: {metrics['v4']['accuracy']:.1%}\n")
    f.write(f"- LLM Accuracy (valid): {metrics['llm']['accuracy_valid']:.1%}\n")
    f.write(f"- LLM Error Rate: {metrics['llm']['error_rate']:.1%}\n")
    f.write(f"- Both Correct: {metrics['comparison']['both_correct_pct']:.1%}\n")
    f.write(f"- Speed Difference: V4 is {llm_lat*1000/v4_lat:.0f}x faster\n")

print(f"Thesis table saved to: {md_path}")

In [ ]:
# Cell 15: Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 1. Accuracy Comparison
ax1 = axes[0]
models = ['V4 Classifier', 'LLM (all)', 'LLM (valid)']
accuracies = [metrics['v4']['accuracy']*100, metrics['llm']['accuracy_overall']*100, metrics['llm']['accuracy_valid']*100]
colors = ['#2196F3', '#FF9800', '#4CAF50']
bars = ax1.bar(models, accuracies, color=colors)
ax1.set_ylabel('Accuracy (%)')
ax1.set_title('Task Accuracy Comparison')
ax1.set_ylim(0, 100)
for bar, acc in zip(bars, accuracies):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{acc:.1f}%', ha='center', fontsize=10)

# 2. Latency Comparison (log scale)
ax2 = axes[1]
latencies = [metrics['v4']['latency_mean_ms'], metrics['llm']['latency_mean_s']*1000]
bars = ax2.bar(['V4 Classifier', 'LLM Extractor'], latencies, color=['#2196F3', '#4CAF50'])
ax2.set_ylabel('Latency (ms)')
ax2.set_title('Latency Comparison')
ax2.set_yscale('log')
for bar, lat in zip(bars, latencies):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.1, f'{lat:.0f}ms', ha='center', fontsize=10)

# 3. Head-to-Head
ax3 = axes[2]
categories = ['Both\nCorrect', 'V4 Only\nCorrect', 'LLM Only\nCorrect', 'Both\nWrong']
counts = [comp['both_correct'], comp['v4_only_correct'], comp['llm_only_correct'], comp['both_wrong']]
colors = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
bars = ax3.bar(categories, counts, color=colors)
ax3.set_ylabel('Number of Queries')
ax3.set_title('Head-to-Head Results')
for bar, cnt in zip(bars, counts):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(cnt), ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'v4_vs_llm_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"\nChart saved to: {OUTPUT_DIR / 'v4_vs_llm_comparison.png'}")

In [ ]:
# Cell 16: Final Summary for Thesis

print("\n" + "=" * 70)
print("THESIS CONCLUSION")
print("=" * 70)

speed_ratio = (metrics['llm']['latency_mean_s']*1000) / metrics['v4']['latency_mean_ms']

print(f"""
## Key Findings

1. **Task Accuracy**:
   - V4 Intent Classifier: {metrics['v4']['accuracy']:.1%}
   - LLM Slot Extraction: {metrics['llm']['accuracy_valid']:.1%} (on valid responses)
   
2. **Response Reliability**:
   - V4: 100% response rate
   - LLM: {(1-metrics['llm']['error_rate']):.1%} response rate ({metrics['llm']['error_rate']:.1%} errors)

3. **Speed**:
   - V4: {metrics['v4']['latency_mean_ms']:.1f}ms average
   - LLM: {metrics['llm']['latency_mean_s']:.2f}s average
   - V4 is {speed_ratio:.0f}x faster

4. **Features**:
   - V4: Task classification only
   - LLM: Task + Entity extraction + Multi-intent + Clarification

## Recommendation

For production e-commerce chatbot:
- Use **V4 for speed-critical** applications (real-time)
- Use **LLM for feature-rich** applications (complex queries)
- Consider **hybrid approach**: V4 for routing, LLM for entity extraction
""")